In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def symbolic_dtft_left_sided():

    omega = sp.symbols('omega', real=True)
    alpha = sp.symbols('alpha', real=True, nonzero=True)

    # ============================================================
    # 1. DEFINITION & CHANGE OF INDEX
    # ============================================================
    q_expr = sp.exp(sp.I * omega) / alpha

    print(r"After the change of index m = -n:")
    display(sp.Eq(sp.Symbol(r'X(e^{j\omega})'), -sp.Sum(q_expr**sp.Symbol('m'), (sp.Symbol('m'), 1, sp.oo))))

    # ============================================================
    # 2. INFINITE GEOMETRIC SERIES SUM
    # ============================================================
    infinite_sum_alpha = q_expr / (1 - q_expr)

    print(r"Infinite sum result:")
    display(infinite_sum_alpha)

    # ============================================================
    # 3. APPLY THE MINUS SIGN FROM x[n]
    # ============================================================
    X_omega = sp.factor(sp.simplify(-infinite_sum_alpha))

    print(r"DTFT Result X(e^{j\omega}):")
    display(X_omega)

    # ============================================================
    # 4. REAL AND IMAGINARY PARTS & MAGNITUDE
    # ============================================================
    X_real = sp.factor(sp.simplify(sp.re(X_omega)))
    X_imag = sp.factor(sp.simplify(sp.im(X_omega)))
    magnitude_sym = sp.factor(sp.simplify(sp.sqrt(X_real**2 + X_imag**2)))

    print(r"Real part:")
    display(X_real)
    print(r"Imaginary part:")
    display(X_imag)
    print(r"Magnitude:")
    display(magnitude_sym)

    # ============================================================
    # 5. NUMERICAL FUNCTIONS
    # ============================================================
    f_real = sp.lambdify((omega, alpha), X_real, modules='numpy')
    f_imag = sp.lambdify((omega, alpha), X_imag, modules='numpy')
    f_mag = sp.lambdify((omega, alpha), magnitude_sym, modules='numpy')

    # ============================================================
    # 6. INTERACTIVE PLOTS
    # ============================================================
    def update_plots(alpha_val):
        clear_output(wait=True)
        omega_vals = np.linspace(-3*np.pi, 3*np.pi, 4000)

        real_vals = np.asarray(f_real(omega_vals, alpha_val), dtype=float)
        imag_vals = np.asarray(f_imag(omega_vals, alpha_val), dtype=float)
        mag_vals = np.asarray(f_mag(omega_vals, alpha_val), dtype=float)
        phase_vals = np.unwrap(np.angle(real_vals + 1j*imag_vals))

        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

        # Magnitude
        axes[0].plot(omega_vals/np.pi, mag_vals, color='red', lw=2.5, label=f'Magnitude, alpha = {alpha_val:.2f}')
        axes[0].set_title(f'DTFT Magnitude Spectrum of $x[n] = -\\alpha^n u[-(n+1)]$  ($\\alpha={alpha_val:.2f}$)', fontsize=11, fontweight='bold')
        axes[0].set_ylabel('Magnitude', fontsize=10)
        axes[0].set_xlim(-3, 3)
        axes[0].set_xticks([-3, -2, -1, 0, 1, 2, 3])
        axes[0].set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        axes[0].grid(True, linestyle='--', alpha=0.6)
        axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

        # Phase
        axes[1].plot(omega_vals/np.pi, phase_vals, color='red', lw=2.5, label=f'Phase, alpha = {alpha_val:.2f}')
        axes[1].set_title('DTFT Phase Spectrum', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Normalized Frequency omega / pi', fontsize=10)
        axes[1].set_ylabel('Phase (radians)', fontsize=10)
        axes[1].set_xlim(-3, 3)
        axes[1].set_xticks([-3, -2, -1, 0, 1, 2, 3])
        axes[1].set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        axes[1].grid(True, linestyle='--', alpha=0.6)
        axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

        plt.subplots_adjust(left=0.08, right=0.80, top=0.93, bottom=0.09, hspace=0.30)
        plt.show()

    alpha_slider = widgets.FloatSlider(
        value=2.0, min=1.05, max=5.0, step=0.05,
        description='Parameter alpha:', style={'description_width':'initial'}
    )

    ui = widgets.VBox([alpha_slider])
    out = widgets.interactive_output(update_plots, {'alpha_val': alpha_slider})

    display(ui, out)

symbolic_dtft_left_sided()